# Spring Petclinic Migration Agent

This notebook implements a LangChain-based migration agent for upgrading Spring Boot 1.5.x to 3.x with JDK 17. It orchestrates the migration process using explicit nodes: executor, build, validate_test_compile, validate_test, and commit_move_up.

Run cells one by one to debug and understand the migration flow.

In [1]:
# Imports and Setup
import os
import subprocess
from pathlib import Path
from typing import List, Optional

# LangChain imports
try:
    from dotenv import load_dotenv
    from langchain_openai import ChatOpenAI
    from langchain_core.tools import tool
    from langgraph.prebuilt import create_react_agent
except ImportError as e:
    raise ImportError(
        "langchain/core/langgraph is required. Install with: pip install langchain langchain-openai langgraph python-dotenv"
    ) from e

load_dotenv(override=True)

/Users/sai/Desktop/Workspace/AI/JDK Migration/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


True

In [2]:
# Constants and Helper Functions
PROJECT_ROOT = Path("/Users/sai/Desktop/Workspace/AI/JDK Migration")
MODULES_LEAF_FIRST = ["petclinic-domain", "petclinic-service", "petclinic-web"]
REWRITE_RECIPE = "org.openrewrite.java.spring.boot3.UpgradeSpringBoot_3_0"
MIGRATION_PLAN_FILE = PROJECT_ROOT / "migration_plan.md"

class CommandError(Exception):
    pass

def load_migration_plan() -> str:
    if not MIGRATION_PLAN_FILE.exists():
        raise FileNotFoundError(f"Migration plan file not found: {MIGRATION_PLAN_FILE}")
    with open(MIGRATION_PLAN_FILE, "r", encoding="utf-8") as f:
        return f.read()

def run_mvn_command(cmd: List[str], cwd: Optional[Path] = None) -> str:
    if cwd is None:
        cwd = PROJECT_ROOT / "spring-petclinic-1.5.x"

    # prefer wrapper in current directory (module) or repo root, else use global mvn
    candidates = [
        cwd / "mvnw",
        PROJECT_ROOT / "mvnw",
        PROJECT_ROOT / "spring-petclinic-1.5.x" / "mvnw",
    ]
    maven_exec = next((p for p in candidates if p.exists()), None)

    if maven_exec is not None:
        full_cmd = [str(maven_exec)] + cmd
        try:
            maven_exec.chmod(0o755)
        except Exception:
            pass
    else:
        full_cmd = ["mvn"] + cmd

    print(f"[CMD] {' '.join(full_cmd)}")
    try:
        completed = subprocess.run(
            full_cmd,
            cwd=cwd,
            capture_output=True,
            text=True,
            check=True,
        )
    except subprocess.CalledProcessError as e:
        reactor_error = "Could not find the selected project in the reactor"
        message = (e.stderr or "") + (e.stdout or "")
        if reactor_error in message and "-pl" in full_cmd:
            print("[WARN] Module not found in reactor; retrying without -pl")
            no_pl_cmd = [token for token in full_cmd if token != "-pl" and not token.startswith(":")]
            print(f"[CMD] {' '.join(no_pl_cmd)}")
            completed = subprocess.run(
                no_pl_cmd,
                cwd=cwd,
                capture_output=True,
                text=True,
                check=True,
            )
            return completed.stdout

        print(f"[ERROR] Command failed with exit {e.returncode}")
        print(e.stdout)
        print(e.stderr)
        raise CommandError((e.returncode, e.stdout, e.stderr))
    return completed.stdout

In [3]:
# Node Functions
def executor_node(module: str) -> str:
    """OpenRewrite AST transform on a module."""
    # Simulate successful rewrite execution
    print(f"Simulating OpenRewrite run for {module} with recipe {REWRITE_RECIPE}")
    return f"Executor node completed for {module}."

def build_node(module: str) -> str:
    """Compile module."""
    # Simulate successful build
    print(f"Simulating clean compile for {module}")
    return f"Build node completed for {module}."

def validate_test_compile_node(module: str) -> str:
    """Run test-compile and return stack trace on failure."""
    # Simulate successful test-compile
    print(f"Simulating test-compile for {module}")
    return f"Validator test-compile node OK for {module}."

def validate_test_node(module: str) -> str:
    """Run module tests."""
    # Simulate successful tests
    print(f"Simulating test run for {module}")
    return f"Validator test node OK for {module}."

def commit_move_up_node(module: str) -> str:
    """Commit and prepare next module."""
    # Simulate commit
    branch = f"migration/{module}".replace("/", "-")
    print(f"Simulating git operations for {module} on branch {branch}")
    return f"Commit & move-up node completed for {module} (branch: {branch})."

In [4]:
# Main Orchestration Function
def run_migration_with_langchain(modules: List[str] = MODULES_LEAF_FIRST) -> None:
    plan_text = load_migration_plan()
    print("\n--- Migration plan loaded from migration_plan.md ---")
    print(plan_text[:1024] + "\n...\n")  # show first chunk for sanity

    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    agent = create_react_agent(
        llm,
        [executor_node, build_node, validate_test_compile_node, validate_test_node, commit_move_up_node],
        prompt="""
You are a migration orchestration agent with an explicit node graph.
For each module, execute nodes in order: executor -> build -> validate_test_compile -> validate_test -> commit_move_up.
Use tools for each step and report status. Do not skip steps.
""",
    )

    for module in modules:
        print(f"\n=== Starting module migration: {module} ===")

        # sequential micro-loop for that module
        result = executor_node(module)
        print(result)

        result = build_node(module)
        print(result)

        result = validate_test_compile_node(module)
        print(result)
        if "failed" in result.lower():
            raise RuntimeError(f"Validator test-compile failed for {module}. Fix and rerun.")

        result = validate_test_node(module)
        print(result)

        result = commit_move_up_node(module)
        print(result)

        print(f"=== Completed module migration: {module} ===")

    print("\nAll module migrations completed successfully.")

In [5]:
# Run the Migration
if __name__ == "__main__":
    run_migration_with_langchain()


--- Migration plan loaded from migration_plan.md ---
# Migration Plan for Spring Boot 1.5.x to Spring Boot 3.x with JDK 17

This migration plan outlines the necessary steps to upgrade the Spring Boot 1.5.x application to Spring Boot 3.x and JDK 17. The plan includes updating dependencies, refactoring code to accommodate package changes from `javax.*` to `jakarta.*`, and ensuring compatibility with the latest versions.

## Step 1: Update Dependencies in `pom.xml`

1. **Update HSQLDB Dependency**
   - **Current Version**: Managed
   - **Recommended Version**: 2.7.4
   - **Action**: Update the version in `pom.xml`.

2. **Update MySQL Connector Dependency**
   - **Current Version**: Managed
   - **Recommended Version**: 8.0.33
   - **Action**: Update the version in `pom.xml`.

3. **Update Cache API Dependency**
   - **Current Version**: Managed
   - **Recommended Version**: 1.1.1
   - **Action**: Consider removing this dependency and using Spring Boot's built-in caching support.

4. **Upd

/var/folders/j4/9sv02n0d3hxdb2cdl4cn6_1c0000gn/T/ipykernel_55147/3703570415.py:8: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(
